### Dallas price inference

This notebook scores listings in **Dallas** - a city the model never saw during training - using the `price_model` saved by the `Texas_price_prediction` notebook.


In [0]:
import json

import pyspark.sql.functions as F
from pyspark.ml import PipelineModel

MODELS_VOLUME = "/Volumes/workspace/default/models"
MODEL_DIR = f"{MODELS_VOLUME}/price_model_airbnb_v1"
BASELINE_DIR = f"{MODELS_VOLUME}/price_model_airbnb_v1_baseline"
PREPROCESSING_JSON = (
    f"{MODELS_VOLUME}/price_model_airbnb_v1_preprocessing.json"
)

CITY_NAME = "dallas"

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
GOLD_FEATURES_TABLE = f"{CATALOG}.default.gold_listings_features"

price_model = PipelineModel.load(MODEL_DIR)

preprocessing_artifacts = json.loads(
    dbutils.fs.head(PREPROCESSING_JSON)
)

median_fill_values = {
    column: float(value)
    for column, value
    in preprocessing_artifacts["median_fill_values"].items()
}

overall_training_median = float(
    preprocessing_artifacts["overall_training_median"]
)

baseline_lookup = spark.read.parquet(BASELINE_DIR)

print(f"Loaded model from {MODEL_DIR} ({len(price_model.stages)} stages)")
print(f"Overall training median: ${overall_training_median:.2f}")


def prepare_baseline_keys(df):
    return (
        df
        .withColumn(
            "_baseline_room_type",
            F.coalesce(F.col("room_type"), F.lit("Unknown"))
        )
        .withColumn(
            "_baseline_accommodates",
            F.coalesce(
                F.col("accommodates").cast("int"),
                F.lit(-1)
            )
        )
    )


def attach_baseline(df):
    keyed_df = prepare_baseline_keys(df)

    return (
        keyed_df
        .join(
            baseline_lookup,
            on=[
                "_baseline_room_type",
                "_baseline_accommodates"
            ],
            how="left"
        )
        .withColumn(
            "baseline_prediction",
            F.coalesce(
                F.col("group_median_price"),
                F.lit(float(overall_training_median))
            )
        )
        .drop(
            "_baseline_room_type",
            "_baseline_accommodates",
            "group_median_price"
        )
    )

Loaded model from /Volumes/workspace/default/models/price_model_airbnb_v1 (10 stages)
Overall training median: $215.00


In [0]:
dallas_df = spark.table(GOLD_FEATURES_TABLE).filter(
    F.col("city") == CITY_NAME
)

dallas_df = dallas_df.withColumn(
    "price", F.col("price").cast("double")
)

boolean_columns = [
    "host_is_superhost",
    "bathroom_shared",
    "is_studio",
    "has_reviews"
]

for c in boolean_columns:
    if c in dallas_df.columns:
        dallas_df = dallas_df.withColumn(c, F.col(c).cast("double"))

categorical_columns = [
    "city",
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

for c in categorical_columns:
    dallas_df = dallas_df.withColumn(c, F.col(c).cast("string"))

numeric_features = list(median_fill_values)

for c in numeric_features:
    dallas_df = dallas_df.withColumn(c, F.col(c).cast("double"))

dallas_model_df = dallas_df.fillna(median_fill_values)

print(f"Dallas listings to score: {dallas_model_df.count():,}")

dallas_predictions = price_model.transform(dallas_model_df).withColumn(
    "model_prediction",
    F.expm1(F.col("model_prediction"))
)

dallas_scored = (
    attach_baseline(dallas_predictions)
    .select(
        "id",
        "city",
        "room_type",
        "accommodates",
        F.col("price").alias("actual_price"),
        "model_prediction",
        "baseline_prediction"
    )
)

display(dallas_scored.limit(10))

Dallas listings to score: 5,913


id,city,room_type,accommodates,actual_price,model_prediction,baseline_prediction
4630130,dallas,Entire home/apt,8.0,477.5,304.7915625994068,326.5
8513649,dallas,Entire home/apt,6.0,228.0,234.98933652759442,266.25
8568055,dallas,Entire home/apt,2.0,106.0,120.12597932813154,143.26
10313293,dallas,Entire home/apt,2.0,84.03,72.82311731657491,143.26
11498438,dallas,Entire home/apt,2.0,91.5,101.65553803557682,143.26
12915291,dallas,Entire home/apt,4.0,220.0,241.99117794807185,198.0
15342315,dallas,Entire home/apt,2.0,91.0,132.23647865164577,143.26
15454467,dallas,Entire home/apt,2.0,179.0,144.95151337722024,143.26
16914852,dallas,Private room,2.0,46.64,146.88069149609484,93.05
19183767,dallas,Private room,2.0,48.0,110.54263469713062,93.05


In [0]:
evaluation_metrics = dallas_scored.agg(
    F.avg(
        F.abs(F.col("actual_price") - F.col("model_prediction"))
    ).alias("model_mae"),
    F.avg(
        F.abs(F.col("actual_price") - F.col("baseline_prediction"))
    ).alias("baseline_mae"),
    F.count("*").alias("total_listings")
).collect()[0]

mae_improvement = (
    100.0
    * (
        evaluation_metrics["baseline_mae"]
        - evaluation_metrics["model_mae"]
    )
    / evaluation_metrics["baseline_mae"]
)

print(f"Dallas listings scored: {evaluation_metrics['total_listings']:,}")
print(f"Dallas Baseline MAE: ${evaluation_metrics['baseline_mae']:.2f}")
print(f"Dallas Model MAE: ${evaluation_metrics['model_mae']:.2f}")
print(f"MAE improvement over baseline: {mae_improvement:.1f}%")

Dallas listings scored: 5,913
Dallas Baseline MAE: $94.40
Dallas Model MAE: $82.34
MAE improvement over baseline: 12.8%


### Save Dallas predictions to Gold

The Dallas predictions are written to `workspace.default.gold_price_predictions` — the table behind the Texas Listings dashboard's **Model performance** page — using the same schema as the training cities' rows, with `split = 'inference'` to mark the holdout-city scored set.

The write uses Delta `replaceWhere` on `city = 'dallas'`, so re-running this notebook replaces only the Dallas rows (never duplicating them) and leaves the austin/fort_worth predictions untouched.

In [0]:
GOLD_PREDICTIONS_TABLE = (
    f"{CATALOG}.default.gold_price_predictions"
)

# snapshot_date is not part of dallas_scored, so it is joined
# back from the source Gold rows.
dallas_predictions_output = (
    dallas_scored
    .join(
        dallas_model_df.select("id", "snapshot_date"),
        on="id",
        how="inner"
    )
    .select(
        "id",
        "city",
        "snapshot_date",
        F.lit("inference").alias("split"),
        "room_type",
        "accommodates",
        "actual_price",
        F.col("model_prediction").alias("predicted_price"),
        F.col("baseline_prediction").alias("baseline_price"),
        F.abs(
            F.col("actual_price") - F.col("model_prediction")
        ).alias("model_absolute_error"),
        F.abs(
            F.col("actual_price") - F.col("baseline_prediction")
        ).alias("baseline_absolute_error"),
        F.lit("airbnb_v1").alias("model_version"),
        F.current_timestamp().alias("prediction_timestamp")
    )
)

# Idempotent write: atomically replace only the Dallas rows.
(
    dallas_predictions_output.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "city = 'dallas'")
    .saveAsTable(GOLD_PREDICTIONS_TABLE)
)

print(f"Dallas predictions saved to {GOLD_PREDICTIONS_TABLE}")

# Validate: city and split counts in the Gold predictions table
saved_city_counts = (
    spark.table(GOLD_PREDICTIONS_TABLE)
    .groupBy("city", "split")
    .count()
    .orderBy("city", "split")
)

city_counts = {
    row["city"]: row["count"]
    for row in (
        spark.table(GOLD_PREDICTIONS_TABLE)
        .groupBy("city")
        .count()
        .collect()
    )
}

assert city_counts.get("dallas") == 5_913, (
    f"Expected 5,913 dallas rows, got {city_counts.get('dallas')}"
)
assert city_counts.get("austin") == 9_759, (
    f"Expected 9,759 austin rows, got {city_counts.get('austin')}"
)
assert city_counts.get("fort_worth") == 2_151, (
    f"Expected 2,151 fort_worth rows, got {city_counts.get('fort_worth')}"
)

null_prediction_rows = (
    spark.table(GOLD_PREDICTIONS_TABLE)
    .filter(F.col("city") == "dallas")
    .filter(
        F.col("predicted_price").isNull()
        | F.col("baseline_price").isNull()
    )
    .count()
)

assert null_prediction_rows == 0, (
    f"Found {null_prediction_rows} Dallas rows with null predictions"
)

print("All validation checks passed.")
print(
    "City counts:",
    {k: v for k, v in sorted(city_counts.items())}
)

display(saved_city_counts)

Dallas predictions saved to workspace.default.gold_price_predictions
All validation checks passed.
City counts: {'austin': 9759, 'dallas': 5913, 'fort_worth': 2151}


city,split,count
austin,test,1997
austin,train,7762
dallas,inference,5913
fort_worth,test,439
fort_worth,train,1712


In [0]:
worst_predictions = (
    dallas_scored
    .withColumn(
        "absolute_error",
        F.abs(
            F.col("actual_price") - F.col("model_prediction")
        )
    )
    .orderBy(F.col("absolute_error").desc())
)

display(worst_predictions)

id,city,room_type,accommodates,actual_price,model_prediction,baseline_prediction,absolute_error
1235446186014942010,dallas,Private room,1.0,1484.0,70.78189971304937,49.0,1413.2181002869506
721671330465940487,dallas,Entire home/apt,4.0,1500.0,259.82477864039197,198.0,1240.175221359608
1678940527114701045,dallas,Entire home/apt,2.0,1264.0,116.02547851658046,143.26,1147.9745214834195
1204281581288123641,dallas,Private room,2.0,1232.0,99.44870289308636,93.05,1132.5512971069136
1630645479093448240,dallas,Entire home/apt,2.0,1294.0,163.78808388162656,143.26,1130.2119161183734
1688577612042934804,dallas,Entire home/apt,6.0,1500.0,392.88469972726364,266.25,1107.1153002727365
1740301073531608186,dallas,Entire home/apt,5.0,1458.2,356.92459425292566,224.5,1101.2754057470743
1685028026370143816,dallas,Entire home/apt,4.0,1215.0,188.97429291070415,198.0,1026.0257070892958
1608060251467712081,dallas,Private room,2.0,1124.0,151.82542720134612,93.05,972.1745727986539
1678342576544488228,dallas,Entire home/apt,2.0,1069.6,116.02547851658046,143.26,953.5745214834194


### Conclusion

The Gradient-Boosted Tree model, trained only on Austin and Fort Worth, generalized well to Dallas — a city it had never seen. Across all 5,913 Dallas listings, the model achieved an MAE of $82.34 versus $94.40 for the median-price baseline, a 12.8% improvement.

This gain is smaller than the 28–34% improvements observed on the held-out test data in the training cities, which is expected: Dallas listings and neighborhoods were entirely unseen during training, so the model could not learn city-specific price patterns. Even so, the model's error on Dallas ($82.34) is comparable to its in-city test errors ($79.75 for Austin, $60.49 for Fort Worth), indicating that the learned relationships between listing characteristics and price transfer reasonably well across Texas cities.

The worst-prediction table above shows that the largest errors come from luxury listings with actual prices well above $1,000 per night, which the model systematically underpredicts. This suggests the training data contains few such extreme listings, and that adding more high-end examples (or a separate luxury segment) could improve coverage at the top of the price range.